# O+O vacuum closure (R4) and the p+p reference for O+O (R3)

Steps R3 and R4 of the plan forward on the tuning page (section *Plan forward*). Produced by
`closure_oo.py` (run + analyse); model results, not vetted physics results.

## Why this test
In the model, $R_{AA}$ divides the O+O spectrum by $N_\mathrm{coll}$ times the p+p spectrum of the **same
chain**. Everything that is not the medium should then cancel. The vacuum closure checks that it does:
O+O with the medium switched off (`--vacuum`: no hydro, no LBT, Matter in vacuum, iColored) against
p+p from the same XML. Any deviation from 1 is a non-medium effect that the O+O $R_{AA}$ would otherwise
silently contain — the nucleons' other collisions, the momentum the hard scattering takes from them
(the hard-momentum subtraction), their remnants and the colour flow into iColored. At 200 GeV a 20 GeV
jet takes a large share of a nucleon's energy, so energy conservation could matter here far more than at
the LHC.

## What the generator does, and so what this test can and cannot see
- Every event has **exactly one hard scattering**, generated by Pythia for a free p+p pair (`Beams:idA/idB =
  2212` in `PythiaIsrGun`), placed at one of the event's binary collisions. Its momentum is subtracted
  from the two nucleons; what 3dMCGlauber leaves of them (the remnants) goes to iColored with the shower.
- So **$N_\mathrm{coll}$ scaling is built in**, and neither isospin nor energy conservation can veto or
  reshape the hard scattering itself. The comparison is per hard scattering: the cross-section-weighted
  spectra $\sum_k \sigma_k/N_k\,w_k$ of O+O and p+p, in the same $\hat p_T$ windows.
- What it **can** see: changes of the hadrons and jets of the hard event from the O+O environment
  (remnants, colour flow, iColored), events that fail, and how often 3dMCGlauber's bookkeeping cannot
  accommodate the hard momentum (its "Can not find x > x_hard" message).
- What it **cannot** see, by construction: the **centrality bias** (with impact-parameter classes and no
  bulk there is no multiplicity to select on; that is R4b), the medium, nuclear PDFs and isospin (a few %
  systematic, quoted separately).

## Settings
`config/OO_MCGlauber_MUSIC_isr_rhictune.xml`: the p+p tune's Pythia lines (`PDF:pSet = 20`,
$\alpha_s(M_Z) = 0.152$ for the hard process and the MPI, MPI `pTmin = 5`, ~1.7 GeV effective) with
Matter $Q_0 = 2$ GeV and `vir_factor` 0.25 — the tune's posterior at $Q_0 = 2$ (`conditional.py`).
p+p: the same XML with 3dMCGlauber's projectile and target set to p (this is also R3, the p+p reference
for O+O). $\hat p_T$ windows 5–80 GeV as in the tune; $\pi^0$ and $\eta$ undecayed; 900k p+p events and
300k O+O events in each impact-parameter class, $b$ = 0–2, 2–4, 4–6, 6–10 fm.

In [ ]:
import json, sys
import numpy as np
import matplotlib.pyplot as plt
import config as C, observables as O, tune as T

data = O.load_data()
S = json.loads((T.RES / "closure_oo.json").read_text())
smp = S["samples"]
U = np.load(T.RES / "closure_oo_units.npz")
CLASSES = [f"OO_b{lo}-{hi}" for lo, hi in S["b_classes"]]
COL = dict(zip(["pp"] + CLASSES, ["k", "C3", "C1", "C2", "C0"]))
LAB = {"pp": "p+p (same XML)", **{c: f"O+O, b {c[4:]} fm" for c in CLASSES}}
obs = {n: np.load(C.PROD_DIR / "obs" / f"closure_{n}.npz") for n in ["pp"] + CLASSES
       if (C.PROD_DIR / "obs" / f"closure_{n}.npz").exists()}
tune_runs = {n: np.load(C.obs_path(n)) for n in ("best", "ref") if C.obs_path(n).exists()}
print(f"{'sample':12s} {'events':>8s} {'<b>':>6s} {'<Ncoll>':>8s} {'<Npart>':>8s} {'<strings>':>9s}"
      f" {'skipped':>8s} {'x_hard msgs':>12s}")
for n in ["pp"] + CLASSES:
    s = smp[n]
    print(f"{n:12s} {int(sum(s['n_events'])):8d} {s['mean_glauber_b']:6.2f} {s['mean_glauber_ncoll']:8.2f}"
          f" {s['mean_glauber_npart']:8.2f} {s['mean_glauber_nstrings']:9.2f} {s['skipped']:8d} {s['xhard_msgs']:12d}")

## R3: the p+p reference in the O+O configuration, against the data
The same observables and $\chi^2$ as in the tune (`tune_comparison.ipynb`): PHENIX $\pi^0$ and STAR jets in
the fit ranges (35 bins, data stat ⊕ sys, normalisation uncertainties as fully correlated), STAR $\pi^0$
as validation. For comparison the tune's validation runs `ref` (0.150, 0.25, 2.0) and `best` (0.153,
0.319, 1.52), made with the p+p template XML. The p+p reference here differs from `ref` only by
$\alpha_s$ 0.152 instead of 0.150 and by coming from the O+O XML.

In [ ]:
runs3 = {"pp (R3)": obs["pp"], **tune_runs}
blocks = []
for o in C.FIT:
    m = T.fit_mask(o, data); d = np.asarray(data[o]["value"])[m]
    rel2 = (np.asarray(data[o]["stat"])[m] ** 2 + np.asarray(data[o]["sys"])[m] ** 2) / d ** 2
    blocks.append((o, m, d, rel2, data[o]["norm"] or 0.0))
for name, ob in runs3.items():
    tot, nb, parts = 0.0, 0, []
    for o, m, d, rel2, norm in blocks:
        v, ve = O.combine(ob, o, data); r = np.log(v[m] / d)
        Cm = np.diag(rel2 + (ve[m] / v[m]) ** 2) + norm ** 2
        c = float(r @ np.linalg.solve(Cm, r)); tot += c; nb += m.sum(); parts.append(f"{o} {c:5.1f}/{m.sum()}")
    print(f"{name:9s} chi2 = {tot:5.1f} / {nb} bins   (" + ", ".join(parts) + ")")

fig, axs = plt.subplots(1, 4, figsize=(18, 3.8))
for ax, (o, pmin) in zip(axs, (("phenix_pi0", 4), ("star_pi0", 4), ("star_jets_eta0", 6), ("star_jets_eta1", 6))):
    e = np.asarray(data[o]["edges"]); x = e.mean(1); d = np.asarray(data[o]["value"])
    err = np.hypot(data[o]["stat"], data[o]["sys"]) / d; sel = x >= pmin
    ax.fill_between(x[sel], 1 - err[sel], 1 + err[sel], color="0.85", step="mid", label="data stat⊕sys")
    for (name, ob), col, ls in zip(runs3.items(), ("C3", "C2", "C0"), ("-", "--", ":")):
        v, ve = O.combine(ob, o, data); m = sel & (v > 0)
        ax.errorbar(x[m], (v / d)[m], (ve / d)[m], fmt=ls, color=col, label=name, lw=1.3)
    if o in C.FIT:
        ax.axvspan(*C.FIT[o], color="C0", alpha=0.05, label="fit range")
    ax.axhline(1, color="k", lw=0.6); ax.set_ylim(0.3, 1.9); ax.set_xlabel(r"$p_T$ [GeV]")
    ax.set_title(data[o]["label"], fontsize=9)
axs[0].set_ylabel("model / data"); axs[0].legend(fontsize=7); fig.tight_layout()

## R4, event level: what O+O does to the hard nucleon pair
Per impact-parameter class (a random 60k-event subsample of each): the impact parameter, $N_\mathrm{coll}$ and
3dMCGlauber's strings (made in vacuum too, though no hydro uses them), and the energy bookkeeping of the
hard pair — the energy the hard scattering takes from the two nucleons (`isr_sub_E_total`: the initiators'
energy after ISR) and what is left of them as remnants for iColored (`remnant_E_proj + remnant_E_targ`). The
subtracted energy is fixed by the hard process and should not depend on the class; the remnants are where
the other collisions of the hard nucleons show up.

In [ ]:
def u(n, k): return U[f"{n}__{k}"].astype(float)
fig, axs = plt.subplots(2, 3, figsize=(16, 7.5))
for n in ["pp"] + CLASSES:
    c = COL[n]; l = LAB[n]
    if n != "pp":
        axs[0, 0].hist(u(n, "glauber_b"), np.linspace(0, 10, 51), histtype="step", color=c, density=True, label=l)
    axs[0, 1].hist(u(n, "glauber_ncoll"), np.arange(0.5, 40.5), histtype="step", color=c, density=True, label=l)
    axs[0, 2].hist(u(n, "glauber_nstrings"), np.arange(-0.5, 60.5), histtype="step", color=c, density=True, label=l)
    rem = u(n, "remnant_E_proj") + u(n, "remnant_E_targ")
    axs[1, 0].hist(rem, np.linspace(0, 200, 81), histtype="step", color=c, density=True, label=l)
    pth = u(n, "pthat"); sub = u(n, "isr_sub_E_total")
    be = np.array([5, 7, 9, 11, 14, 18, 23, 30, 40, 55, 80.])
    idx = np.digitize(pth, be) - 1
    xc = 0.5 * (be[:-1] + be[1:])
    for ax, val in ((axs[1, 1], sub), (axs[1, 2], rem)):
        mean = [np.nanmean(val[idx == i]) if (idx == i).sum() > 20 else np.nan for i in range(len(xc))]
        ax.plot(xc, mean, "o-", color=c, ms=3, label=l)
axs[0, 0].set_xlabel("b [fm]"); axs[0, 1].set_xlabel(r"$N_\mathrm{coll}$"); axs[0, 1].set_yscale("log")
axs[0, 2].set_xlabel("3dMCGlauber strings"); axs[0, 2].set_yscale("log")
axs[1, 0].set_xlabel("hard pair: remnant energy (proj + targ) [GeV]")
axs[1, 1].set_xlabel(r"$\hat p_T$ [GeV]"); axs[1, 1].set_ylabel("subtracted energy (initiators) [GeV]")
axs[1, 2].set_xlabel(r"$\hat p_T$ [GeV]"); axs[1, 2].set_ylabel("hard pair: remnant energy [GeV]")
for ax in axs.flat: ax.legend(fontsize=7)
for ax in (axs[1, 1], axs[1, 2]): ax.set_xscale("log")
fig.tight_layout()

## R4, the closure: O+O / p+p per hard scattering
Top: the spectra (cross section per hard scattering, the same windows and normalisation as the tune).
Bottom: O+O / p+p per class; 1 means the O+O environment leaves the hard event's hadrons and jets
unchanged. $\pi^0$ in PHENIX's binning ($|y|<0.35$), charged hadrons ($|\eta|<1$), STAR's jets
($R = 0.5$, off-axis-cone UE subtraction) at $|\eta|<0.5$ and $0.5<|\eta|<0.9$. Statistical errors only.

In [ ]:
def charged(n):
    s = smp[n]; return np.asarray(s["ch"]), np.asarray(s["ch_err"])
che = np.asarray(S["ch_edges"])
panels = [("phenix_pi0", r"$\pi^0$, $|y|<0.35$ (PHENIX binning)", 4),
          ("charged", r"charged hadrons, $|\eta|<1$", 2),
          ("star_jets_eta0", r"jets $R=0.5$, $|\eta|<0.5$", 6),
          ("star_jets_eta1", r"jets $R=0.5$, $0.5<|\eta|<0.9$", 6)]
def spec(n, o):
    if o == "charged":
        v, e = charged(n); x = 0.5 * (che[:-1] + che[1:]); return x, v, e
    ee = np.asarray(data[o]["edges"]); v, e = O.combine(obs[n], o, data); return ee.mean(1), v, e
fig, axs = plt.subplots(2, 4, figsize=(19, 7), gridspec_kw={"height_ratios": [1.6, 1]}, sharex="col")
summary_ratio = {}
for k, (o, title, pmin) in enumerate(panels):
    x, vp, ep = spec("pp", o); sel = (x >= pmin) & (vp > 0)
    axs[0, k].errorbar(x[sel], vp[sel], ep[sel], fmt="k-", lw=1.4, label=LAB["pp"])
    for n in CLASSES:
        _, v, e = spec(n, o); m = sel & (v > 0)
        axs[0, k].errorbar(x[m], v[m], e[m], fmt="o", ms=3, color=COL[n], label=LAB[n])
        r = v / vp; re = r * np.hypot(e / np.where(v > 0, v, 1), ep / vp)
        axs[1, k].errorbar(x[m], r[m], re[m], fmt="o-", ms=3, color=COL[n], lw=0.8)
        w = 1 / re[m] ** 2
        summary_ratio[(o, n)] = (np.sum(w * r[m]) / np.sum(w), 1 / np.sqrt(np.sum(w)))
    axs[0, k].set_yscale("log"); axs[0, k].set_title(title, fontsize=10)
    axs[1, k].axhline(1, color="k", lw=0.6); axs[1, k].set_ylim(0.8, 1.2)
    axs[1, k].set_xlabel(r"$p_T$ [GeV]"); axs[1, k].set_xscale("log")
axs[1, 0].set_ylabel("O+O / p+p (per hard scattering)"); axs[0, 0].legend(fontsize=7)
axs[0, 0].set_ylabel(r"$E\,d^3\sigma/dp^3$ [mb GeV$^{-2}$]"); axs[0, 1].set_ylabel(r"$d^2\sigma/dp_Td\eta$ [mb/GeV]")
axs[0, 2].set_ylabel(r"$d^2\sigma/dp_Td\eta$ [pb/GeV]")
fig.tight_layout()
print(f"{'O+O / p+p, weighted mean over the plotted range':48s}" + "".join(f"{c:>16s}" for c in CLASSES))
for o, title, _ in panels:
    print(f"{o:48s}" + "".join(f"{summary_ratio[(o, c)][0]:9.3f}±{summary_ratio[(o, c)][1]:.3f}" for c in CLASSES))

### Cross-check at the parton level
The hard process itself should be identical in O+O and p+p (the same Pythia, the same windows): the
cross-section-weighted spectrum of the leading outgoing parton of the hard scattering (`parton_pt_lead`, before
the final-state shower) and its rapidity, from the 60k-event subsamples. A deviation here would point to the
event bookkeeping (windows, weights, skipped events), not to the remnants or the hadronization.

In [ ]:
be = np.array([5, 6, 7, 8, 9, 10, 11, 12.5, 14, 16, 18, 20.5, 23, 26.5, 30, 35, 40, 47.5, 55, 67, 80])
fig, axs = plt.subplots(1, 2, figsize=(12, 3.8))
sig = {n: np.asarray(smp[n]["sigma"]) for n in ["pp"] + CLASSES}
def wspec(n):
    pt, wi = u(n, "parton_pt_lead"), u(n, "pthat_bin").astype(int)
    nk = np.bincount(wi, minlength=len(sig[n])); w = sig[n][wi] / nk[wi]
    h = np.histogram(pt, be, weights=w)[0]; h2 = np.histogram(pt, be, weights=w ** 2)[0]
    return h / np.diff(be), np.sqrt(h2) / np.diff(be)
hp, ep = wspec("pp"); xc = 0.5 * (be[:-1] + be[1:])
for n in CLASSES:
    h, e = wspec(n); r = h / hp
    axs[0].errorbar(xc, r, r * np.hypot(e / h, ep / hp), fmt="o-", ms=3, color=COL[n], label=LAB[n])
    axs[1].hist(u(n, "parton_y_lead"), np.linspace(-3, 3, 41), histtype="step", density=True, color=COL[n], label=LAB[n])
axs[1].hist(u("pp", "parton_y_lead"), np.linspace(-3, 3, 41), histtype="step", density=True, color="k", lw=1.5, label=LAB["pp"])
axs[0].axhline(1, color="k", lw=0.6); axs[0].set_ylim(0.8, 1.2); axs[0].set_xscale("log")
axs[0].set_xlabel(r"leading hard parton $p_T$ [GeV]"); axs[0].set_ylabel("O+O / p+p (σ-weighted)")
axs[1].set_xlabel("leading hard parton $y$"); axs[1].set_ylabel("normalised")
for ax in axs: ax.legend(fontsize=7)
fig.tight_layout()

## Where the hard event's particles go: $dN_{ch}/d\eta$ and the remnants
Full-$\eta$ samples (no $\eta$ cut, iColored's input stored; 20k events at $\hat p_T$ 9–11 and 23–30 GeV per
class): the charged particles of the hard event (shower + remnants through iColored; there is no bulk in
vacuum), and the rapidity of the partons iColored fragments. In O+O the hard nucleons have had other
collisions, so their remnants carry less energy; this is where a difference from p+p is expected, and it
matters for R4b, since forward multiplicity is what STAR's centrality estimators count.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 4))
for n in ["pp"] + CLASSES:
    s = smp.get(n + "_eta", {}).get("eta")
    if not s: continue
    ed = np.asarray(s["edges"]); xc = 0.5 * (ed[:-1] + ed[1:])
    lw = 1.6 if n == "pp" else 1.1
    axs[0].plot(xc, s["9-11"]["dndeta"], color=COL[n], lw=lw, label=LAB[n])
    axs[1].plot(xc, s["23-30"]["dndeta"], color=COL[n], lw=lw, label=LAB[n])
    axs[2].plot(xc, s["9-11"]["parton_dndy"], color=COL[n], lw=lw, label=LAB[n])
for ax, t in zip(axs, (r"$\hat p_T$ 9–11 GeV", r"$\hat p_T$ 23–30 GeV", r"$\hat p_T$ 9–11 GeV")):
    ax.set_xlim(-7, 7); ax.set_ylim(0); ax.set_title(t, fontsize=9); ax.legend(fontsize=7)
    ax.axvspan(2.1, 5.1, color="C0", alpha=0.06); ax.axvspan(-5.1, -2.1, color="C0", alpha=0.06)
axs[0].set_ylabel(r"hard event: $dN_{ch}/d\eta$ per event"); axs[0].set_xlabel(r"$\eta$"); axs[1].set_xlabel(r"$\eta$")
axs[2].set_xlabel("parton rapidity $y$"); axs[2].set_ylabel("iColored input: partons $dN/dy$")
fig.tight_layout()
print("shaded: 2.1 < |eta| < 5.1, the acceptance of STAR's EPD (forward centrality estimator)")
for n in ["pp"] + CLASSES:
    s = smp.get(n + "_eta", {}).get("eta")
    if not s: continue
    ed = np.asarray(s["edges"]); xc = 0.5 * (ed[:-1] + ed[1:]); w = np.diff(ed)
    for win in ("9-11", "23-30"):
        y = np.asarray(s[win]["dndeta"])
        print(f"{n:12s} {win:6s} dN/deta |eta|<1: {y[np.abs(xc) < 1].mean():5.2f}   "
              f"N in 2.1<|eta|<5.1: {np.sum((y * w)[(np.abs(xc) > 2.1) & (np.abs(xc) < 5.1)]):5.2f}   "
              f"total: {np.sum(y * w):5.1f}")

## Results and what they mean
Model results (this notebook's runs: 900k p+p and 4 × 300k O+O events in vacuum, $\hat p_T$ 5–80 GeV;
statistical errors only), not vetted physics results.

**R3, the p+p reference in the O+O configuration.** The jets agree with the tune's runs and the data as
before (jets $\chi^2$ 7.8 + 4.9 / 22 bins, against 9.8 + 4.4 for `ref`); the jets are ~2–3 % above `ref`,
as expected from $\alpha_s$ 0.152 instead of 0.150 ($\sigma\propto\alpha_s^2$), so the p+p version of the O+O XML
runs the same chain as the tune's template. The total $\chi^2$ is 42.2 / 35, against 24.5 for `ref` and 36.1
for `best`; the difference is in the PHENIX $\pi^0$ (29.5 / 13 against 10.3 and 20.5), where all three runs
show the known shape tension (model harder than PHENIX at 14–18 GeV) and the model's $\pi^0$ statistics are
poor (8–20 % per bin). The spread between runs that hardly differ (10–30 for the $\pi^0$) says the model's
$\pi^0$ errors are underestimated or the statistics too small to rank the runs on the $\pi^0$; extra
low-window events (as for the design points) would settle it. As the p+p reference for O+O $R_{AA}$ this run
is adequate: the same chain as O+O, matching the data where the tune does.

**R4, the vacuum closure.**
- **The hard process is the same in O+O and p+p** (leading-parton spectrum and rapidity: ratio 1 within
  the statistics), and so is the energy it takes from the two nucleons (the subtracted energy vs $\hat p_T$
  is identical in all classes), as built.
- **The hard yields close.** Per hard scattering, O+O / p+p is 1.00–1.02 for jets in both $\eta$ ranges in
  every class; the charged hadrons at $|\eta|<1$ are 1.00 (peripheral) to 1.03 (central, up to +5–8 % at
  3–6 GeV); the $\pi^0$ is 0.90–1.00 with ±4 % errors (consistent with 1 at 1–3 σ). The small central
  excess of charged hadrons is a hadronization effect of the weaker remnants (below), not of the hard
  process.
- **Failures are negligible:** 13 skipped events out of 300k in central O+O (16 in all classes), and no
  "Can not find x > x_hard" message with the tune's MPI (the earlier messages came with full soft MPI).
- **Where O+O differs: the hard nucleons' remnants.** In central O+O the hard pair's remnant energy is
  ~65 GeV instead of ~115 GeV in p+p at $\hat p_T$ 10 GeV (their other collisions' strings take the rest); the
  remnant partons sit at lower rapidity, and the hard event puts ~20 % fewer charged particles into
  $2.1<|\eta|<5.1$ (STAR's EPD acceptance: 9.6 instead of 12.0 per event at $\hat p_T$ 9–11 GeV), with the
  mid-rapidity yield unchanged.

**What it means.** In this chain energy conservation does not touch the hard yield at fixed impact parameter
(by construction, and the closure confirms that nothing else does, within ~2 % for jets and ~3 % for charged
hadrons); N$_\mathrm{coll}$ scaling holds per hard scattering, and the O+O $R_{AA}$ at fixed geometry measures the
medium. Energy conservation acts on the soft side instead: the hard nucleons carry less energy forward. That
is the ingredient of the centrality bias (R4b): with hydro, the subtraction also weakens the hard nucleons'
strings, and jet events then put less multiplicity into a forward estimator — the effect R4b has to
quantify with multiplicity classes. This vacuum test shows the hard event's own share of it: −20 % in the
EPD acceptance in central O+O, a small part of the total forward multiplicity there.
